# Public Benchmark: Best DME Forecast-Pretrain Architecture


In [ ]:
# Cell 1 - Setup & Install
import json
import os
import pickle
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import yaml
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyyaml", "-q"])
    import yaml

REPO_DIR = Path("/kaggle/working/denoising-event-sequences")
if not REPO_DIR.exists():
    REPO_DIR = Path.cwd()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

DATA_ROOT = Path("/kaggle/working/data")
RAW_ROOT = DATA_ROOT / "raw"
PROCESSED_ROOT = DATA_ROOT / "processed"
OUTPUT_ROOT = Path("/kaggle/working/outputs/public_benchmarks")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def run_logged(cmd, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("Running:", " ".join(map(str, cmd)))
    with log_path.open("w") as log_file:
        proc = subprocess.Popen(
            [str(x) for x in cmd],
            cwd=str(REPO_DIR),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="")
            log_file.write(line)
        return_code = proc.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)
    print(f"Log saved: {log_path}")


In [ ]:
# Shared - Prepare Public Benchmark Data If Missing
DATASET_NAME = "gender"  # "gender" or "age_group"
DATASET_CONFIG_PATHS = {
    "gender": REPO_DIR / "configs" / "datasets" / "gender.yaml",
    "age_group": REPO_DIR / "configs" / "datasets" / "age_group.yaml",
}
PROCESSED_DIR = PROCESSED_ROOT / f"{DATASET_NAME}_benchmark"
RUN_PREPARE_DATA_IF_MISSING = True
FORCE_REBUILD_PROCESSED = False
MAX_SEQ_LEN = 512

required_artifacts = ["events.parquet", "canonical_events.parquet", "splits.json", "preprocessor.pkl", "prepared_config.yaml", "data_report.json"]
artifacts_exist = all((PROCESSED_DIR / name).exists() for name in required_artifacts)
if FORCE_REBUILD_PROCESSED and PROCESSED_DIR.exists():
    shutil.rmtree(PROCESSED_DIR)
    artifacts_exist = False

if RUN_PREPARE_DATA_IF_MISSING and not artifacts_exist:
    run_logged(
        [
            sys.executable,
            REPO_DIR / "scripts" / "prepare_public_benchmark_data.py",
            "--dataset", DATASET_NAME,
            "--raw-root", RAW_ROOT,
            "--output-dir", PROCESSED_DIR,
            "--max-seq-len", MAX_SEQ_LEN,
        ],
        OUTPUT_ROOT / "logs" / f"{DATASET_NAME}_prepare_public_benchmark_data.log",
    )

assert all((PROCESSED_DIR / name).exists() for name in required_artifacts), PROCESSED_DIR
with (PROCESSED_DIR / "data_report.json").open() as f:
    data_report = json.load(f)
print(json.dumps(data_report, indent=2)[:3000])


In [ ]:
# Cell 4 - Final DME Sweep Configs
RUN_FORECAST_PRETRAIN = True
RUN_FINETUNE = True
SMOKE_RUN = False

BASE_CONFIG_PATH = REPO_DIR / "configs" / "base.yaml"
DATASET_CONFIG_PATH = DATASET_CONFIG_PATHS[DATASET_NAME]
FINAL_DME_ABLATION_PATH = REPO_DIR / "configs" / "ablations" / "A14_final_forecast_pretrain.yaml"
FINAL_DME_PUBLIC_EXPERIMENTS = {
    # Best Rosbank setting from the previous sweep, translated to valid A14 keys.
    "forecast_alpha030_reg012": {
        "forecasting": {"alpha_forecast": 0.30, "alpha_denoising": 0.30},
        "model": {"dropout": 0.12},
        "training": {"weight_decay": 0.012, "num_epochs_pretrain": 20 if SMOKE_RUN else 120, "num_epochs_finetune": 4 if SMOKE_RUN else 40},
    },
    "forecast_alpha030_reg015": {
        "forecasting": {"alpha_forecast": 0.30, "alpha_denoising": 0.30},
        "model": {"dropout": 0.15},
        "training": {"weight_decay": 0.015, "num_epochs_pretrain": 20 if SMOKE_RUN else 120, "num_epochs_finetune": 4 if SMOKE_RUN else 40},
    },
}


In [ ]:
# Cell 5 - Write Runtime Configs
CONFIG_DIR = OUTPUT_ROOT / DATASET_NAME / "final_dme" / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

def deep_update(a, b):
    out = dict(a)
    for k, v in b.items():
        out[k] = deep_update(out[k], v) if isinstance(out.get(k), dict) and isinstance(v, dict) else v
    return out

with BASE_CONFIG_PATH.open() as f:
    base_cfg = yaml.safe_load(f)
with DATASET_CONFIG_PATH.open() as f:
    dataset_cfg = yaml.safe_load(f)
with FINAL_DME_ABLATION_PATH.open() as f:
    ablation_cfg = yaml.safe_load(f)

num_classes = 2 if DATASET_NAME == "gender" else 4
selection_metric = "roc_auc" if DATASET_NAME == "gender" else "macro_f1"
runtime_config_paths = {}
for exp_name, patch in FINAL_DME_PUBLIC_EXPERIMENTS.items():
    cfg = deep_update(deep_update(base_cfg, dataset_cfg), ablation_cfg)
    cfg = deep_update(cfg, patch)
    cfg["data"]["timestamp_col"] = "event_timestamp"
    cfg["data"]["max_seq_len"] = 512
    cfg["training"].update({
        "num_classes": num_classes,
        "task": "binary" if num_classes == 2 else "multiclass",
        "selection_metric": selection_metric,
        "early_stopping_patience": 2 if SMOKE_RUN else 8,
    })
    path = CONFIG_DIR / f"{exp_name}.yaml"
    with path.open("w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    runtime_config_paths[exp_name] = path
runtime_config_paths


In [ ]:
# Cell 6 - Forecast Pretraining
for exp_name, config_path in runtime_config_paths.items():
    out_dir = OUTPUT_ROOT / DATASET_NAME / "final_dme" / exp_name / "forecast_pretrain"
    ckpt = out_dir / "checkpoints" / "best_forecast_checkpoint.pt"
    if RUN_FORECAST_PRETRAIN and not ckpt.exists():
        run_logged(
            [
                sys.executable,
                REPO_DIR / "scripts" / "forecast_pretrain.py",
                "--config", config_path,
                "--data-dir", PROCESSED_DIR,
                "--output-dir", out_dir,
            ],
            OUTPUT_ROOT / "logs" / f"{DATASET_NAME}_{exp_name}_forecast_pretrain.log",
        )
    assert ckpt.exists(), ckpt


In [ ]:
# Cell 7 - Fine-tune Full Encoder
rows = []
for exp_name, config_path in runtime_config_paths.items():
    pretrain_ckpt = OUTPUT_ROOT / DATASET_NAME / "final_dme" / exp_name / "forecast_pretrain" / "checkpoints" / "best_forecast_checkpoint.pt"
    out_dir = OUTPUT_ROOT / DATASET_NAME / "final_dme" / exp_name / "finetune_full"
    if RUN_FINETUNE:
        run_logged(
            [
                sys.executable,
                REPO_DIR / "scripts" / "finetune.py",
                "--config", config_path,
                "--pretrained-checkpoint", pretrain_ckpt,
                "--data-dir", PROCESSED_DIR,
                "--output-dir", out_dir,
            ],
            OUTPUT_ROOT / "logs" / f"{DATASET_NAME}_{exp_name}_finetune_full.log",
        )
    metrics_files = sorted((out_dir / "metrics").glob("*_finetune_metrics.json"))
    with metrics_files[-1].open() as f:
        metrics = json.load(f)["test_metrics"]
    rows.append({"experiment": exp_name, "pipeline": "full_finetune", **metrics})

final_dme_metrics_df = pd.DataFrame(rows).sort_values("macro_f1", ascending=False)
metrics_csv = OUTPUT_ROOT / DATASET_NAME / "final_dme_metrics.csv"
final_dme_metrics_df.to_csv(metrics_csv, index=False)
print("Metrics CSV:", metrics_csv)
final_dme_metrics_df


In [ ]:
# Cell 8 - Forecast Generation / Scenario Quality Metrics
forecast_rows = []
for exp_name in runtime_config_paths:
    metrics_path = OUTPUT_ROOT / DATASET_NAME / "final_dme" / exp_name / "forecast_pretrain" / "metrics" / "forecast_eval_metrics.json"
    scenario_path = OUTPUT_ROOT / DATASET_NAME / "final_dme" / exp_name / "forecast_pretrain" / "metrics" / "scenario_examples.json"
    if metrics_path.exists():
        with metrics_path.open() as f:
            forecast_rows.append({"experiment": exp_name, **json.load(f)})
        print(f"[{exp_name}] Scenario examples:", scenario_path)
forecast_metrics_df = pd.DataFrame(forecast_rows)
forecast_metrics_df.to_csv(OUTPUT_ROOT / DATASET_NAME / "final_dme_forecast_metrics.csv", index=False)
forecast_metrics_df


In [ ]:
# Cell 9 - Artifact Summary
print("Processed:", PROCESSED_DIR)
print("DME outputs:", OUTPUT_ROOT / DATASET_NAME / "final_dme")
print("Best by macro_f1:")
final_dme_metrics_df.head(1)
